In [1]:
# Requisitos (descomenta si no lo tienes instalado):
# %pip install azure-storage-blob

import os
import sys
import json

# extract_search_text.py vive en ../tools respecto a este notebook (testfiles/).
TOOLS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "tools"))
if TOOLS_DIR not in sys.path:
    sys.path.insert(0, TOOLS_DIR)

from extract_search_text import extract_search_text
from azure.storage.blob import BlobServiceClient

ModuleNotFoundError: No module named 'azure'

In [ ]:
# --- Conexion a Azure Blob Storage ---
# La cadena de conexion se lee de una variable de entorno (no la escribas aqui).
CONNECTION_STRING = os.environ.get("AZURE_STORAGE_CONNECTION_STRING", "<pega-aqui-tu-connection-string>")

# ORIGEN: contenedor 'data', carpeta con los JSON a procesar.
SOURCE_CONTAINER = "data"
SOURCE_PREFIX = "products/products_metadata/"

# DESTINO: contenedor distinto donde se dejaran los .md generados.
DEST_CONTAINER = "markdown-test2"
DEST_PREFIX = ""  # raiz del contenedor destino; pon "subcarpeta/" si quieres agruparlos

In [ ]:
# --- Funciones auxiliares ---

def get_container_client(container_name):
    """Cliente de un contenedor a partir de la cadena de conexion."""
    service = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    return service.get_container_client(container_name)


def list_source_blobs(container):
    """Nombres de los blobs .json bajo SOURCE_PREFIX."""
    return [b.name for b in container.list_blobs(name_starts_with=SOURCE_PREFIX)
            if b.name.lower().endswith(".json")]


def dest_name_for(blob_name):
    """data/products/products_metadata/5A154BC00.json -> <DEST_PREFIX>5A154BC00.md"""
    base = os.path.splitext(os.path.basename(blob_name))[0]
    return f"{DEST_PREFIX}{base}.md"


def process_blob(src_container, dest_container, blob_name):
    """Descarga el JSON del origen, extrae el texto y sube el .md al destino. Devuelve el nombre destino."""
    raw = src_container.download_blob(blob_name).readall()
    product = json.loads(raw)
    text = extract_search_text(product)
    dest = dest_name_for(blob_name)
    dest_container.upload_blob(name=dest, data=text.encode("utf-8"), overwrite=True)
    return dest

In [ ]:
# --- Main: itera sobre los ficheros del origen y sube el resultado al destino, mostrando el progreso ---
src_container = get_container_client(SOURCE_CONTAINER)
dest_container = get_container_client(DEST_CONTAINER)

blobs = list_source_blobs(src_container)
print(f"Encontrados {len(blobs)} ficheros JSON en '{SOURCE_CONTAINER}/{SOURCE_PREFIX}'\n")

ok, errores = 0, 0
for i, blob_name in enumerate(blobs, 1):
    try:
        dest = process_blob(src_container, dest_container, blob_name)
        ok += 1
        print(f"[{i}/{len(blobs)}] OK    {blob_name} -> {DEST_CONTAINER}/{dest}")
    except Exception as e:
        errores += 1
        print(f"[{i}/{len(blobs)}] ERROR {blob_name}: {e}")

print(f"\nHecho. {ok} subidos, {errores} con error.")